# 🧠 Synthetic Instruction Dataset for LLM Fine-Tuning

In this lesson, we are going to generate a synthetic instruction dataset — question-and-answer pairs grounded in real documents, that can be used to fine-tune or evaluate a large language model without exposing the source content.

In this notebook, we will:
1. Generate Q&A pairs from a folder of documents using `DocumentQAGeneration`
2. Inspect and quality-filter the output

## 1. Generate Q&A Pairs

`generate()` accepts a folder path, a single file path, or a PyArrow table as `input_source`. `num_qa_pairs` sets how many pairs to generate **per document**.

In [ ]:
#add here the subcription key
subscription_key='add-your-subscription-key' 

In [ ]:
from ydata.synthesizers.text.model.qa import DocumentQAGeneration

qa_generator = DocumentQAGeneration(
    backend='openai',
    subscription_key=subscription_key
)

print('DocumentQAGeneration ready ✓')

In [ ]:
# Generate Q&A pairs from a folder of documents
# Replace 'assets/documents/' with the path to your own document folder
result = qa_generator.generate(
    input_source='assets/documents/',   # folder, single file, or PyArrow table
    docs_extension='docx',              # 'docx' | 'txt' | 'pdf'
    num_qa_pairs=5,                     # pairs per document
)

print(f'Generated {len(result)} Q&A pairs ✓')

In [ ]:
print(f'Shape: {result.shape}')
print(f'Columns: {list(result.columns)}')
result.head()

In [ ]:
# Distribution of Q&A pairs across source documents
print('Pairs per source document:')
print(result['source'].value_counts().to_string())

In [ ]:
# Spot-check a single Q&A pair
sample = result.sample(1).iloc[0]
print(f'Source  : {sample["source"]}')
print(f'Question: {sample["question"]}')
print(f'Answer  : {sample["answer"]}')

## 2. Export to JSONL for Fine-Tuning

Instruction fine-tuning pipelines expect data in JSONL format — one JSON object per line. `question` maps to `instruction` and `answer` maps to `response`.

In [ ]:
output_path = 'output/instruction_dataset.jsonl'
os.makedirs('output', exist_ok=True)

with open(output_path, 'w') as f:
    for _, row in filtered.iterrows():
        record = {
            'instruction': row['question'],
            'response':    row['answer'],
            'source':      row['source'],
        }
        f.write(json.dumps(record) + '\n')

print(f'Exported {len(filtered)} pairs → {output_path}')

In [ ]:
# Preview the first three records
with open(output_path) as f:
    for i, line in enumerate(f):
        if i >= 3:
            break
        print(json.dumps(json.loads(line), indent=2))
        print()

## ✅ Summary

In this lesson you:

1. **Configured an LLM backend** — wired an Anthropic or OpenAI key into the environment so `DocumentQAGeneration` can call it internally
2. **Generated Q&A pairs** using `DocumentQAGeneration.generate()`, pointed at a folder of documents with a specified file type and pairs-per-document count
3. **Inspected the output** — a DataFrame with `question`, `answer`, and `source` columns, traceable back to each source file

The key insight: synthetic instruction datasets are only as good as their source documents. Combining `DocumentGenerator` (from the previous video) with `DocumentQAGeneration` gives you a fully synthetic pipeline — generate the documents, then generate the instruction dataset from them.